__Пример реализации RAG__ 

Евгений Борисов <esborisov@sevsu.ru>

# загружаем текст

In [1]:
import gzip
import requests
from bs4 import BeautifulSoup

In [2]:
# Василий Аксенов. Затоваренная бочкотара

url = 'https://www.booksite.ru/fulltext/0/001/005/028/026.htm'
source = requests.get(url)
source.encoding='windows-1251' 
text = BeautifulSoup(source.text).get_text()
print( text.strip()[:524] )

# with gzip.open('bochkotara.txt.gz','rt') as f: text = f.read()
with gzip.open('bochkotara.txt.gz','wt') as f: f.write(text)

Василий Аксенов. Затоваренная бочкотара
Василий Аксенов. Затоваренная бочкотара


 Журнальный вариант. Печатался в журнале "Юность"
 Spellchecked by Tanya Andrushchenko

Повесть с преувеличениями и сновидениями


                                    Затоварилась бочкотара, зацвела желтым
                                 цветком, затарилась, затюрилась и с места
                                 стронулась.
                                                                 Из газет.


     В палисаднике под 


# режем текст на чанки

In [3]:
# pip install langchain langchain-community

In [4]:
import langchain
langchain.__version__

'1.2.0'

In [5]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=30)
chunked_docs = splitter.split_text(text)

In [6]:
display( len(chunked_docs) )
print(chunked_docs[2] )

308

назад грузовика.
     Хозяин - потомственный рабочий пенсионного  возраста,  тихо  и  уютно
сидящий  на  скамейке  с  цигаркою  в  желтых,  трудно  зажатых   пальцах,
рассказывает приятелю, почти двойнику, о художествах сына:
     - Я совсем атрофировал к нему отцовское отношение.  Мы,  Телескоповы,
сам знаешь, Петр Ильич, по механической части, в лабораторных цехах, слуги
индустрии. В четырех коленах, Петр Ильич, как знаешь.  Сюда,  к  идиотизму


# строим векторное хранилище

In [ ]:
# EMB_MODEL_ID = 'sentence-transformers/all-minilm-l12-v2'
# EMB_MODEL_ID = 'google/gemini-embedding-001'
# EMB_MODEL_ID = 'mistralai/mistral-embed-2312'
# EMB_MODEL_ID = 'openai/text-embedding-3-large'
EMB_MODEL_ID = 'qwen/qwen3-embedding-0.6B'
# EMB_MODEL_ID = 'qwen/qwen3-embedding-4b'
# EMB_MODEL_ID = 'qwen/qwen3-embedding-8b'

In [ ]:
# pip install langchain_huggingface

In [ ]:
# from langchain_huggingface.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL_ID)

In [ ]:
# pip install langchain-openai

In [ ]:
from langchain_openai import OpenAIEmbeddings
from keys import OPENROUTER_API_KEY

embeddings = OpenAIEmbeddings(
      model=EMB_MODEL_ID,
      openai_api_base='https://openrouter.ai/api/v1', 
      openai_api_key=OPENROUTER_API_KEY
    )

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_texts(chunked_docs, embeddings)

In [ ]:
retriever = db.as_retriever( search_type='similarity',  search_kwargs={'k': 4} )

# LLM для обработки результатов поиска

In [9]:
# pip install langchain_openai

In [12]:
from langchain_openai import ChatOpenAI
from keys import OPENROUTER_API_KEY

MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b:free"

llm = ChatOpenAI(
    model=MODEL_ID, # Или любая другая модель, поддерживаемая OpenRouter
    openai_api_base='https://openrouter.ai/api/v1', 
    openai_api_key=OPENROUTER_API_KEY
)

In [21]:
query = 'что делают скопления пчел?'

In [22]:
response = llm.invoke(query + " Ответ должен быть кратким до 30 слов.")
print(response.content)

Скопления пчел собирают нектар и пыльцу, превращают их в мед, поддерживают температуру в улье, защищают матку и распределяют пищу. Это база жизни колонии.


# собираем всё вместе

In [ ]:
from langchain_classic.chains import RetrievalQA

retriever = vector_store.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever
    )

In [ ]:
response = qa_chain.invoke(query)
print("Ответ:", response["result"])